In [ ]:
import time
import numpy as np
import scipy.io
import scipy.sparse as sp

from leja_logdet import (
    FocalIntervalEstimator, DividedDifferencesLog, LejaLogAction, HutchPP, LogDetEstimatorHutchPPLeja
)

def load_uf_matrix_mat(file_path: str):
    data = scipy.io.loadmat(file_path)
    problem = data["Problem"]
    A = problem["A"][0, 0]
    # UF .mat often stores sparse as scipy sparse already; enforce CSR
    if sp.issparse(A):
        return A.tocsr()
    return sp.csr_matrix(A)

if __name__ == "__main__":
    Q = load_uf_matrix_mat("ship_001.mat")
    n = Q.shape[0]
    print("Loaded Q:", Q.shape, "nnz =", Q.nnz)

    m_leja = 300
    m_hutchpp = 60
    tol = 1e-6

    Leja_X_all = np.loadtxt("Leja_10000.txt")
    Leja_X = Leja_X_all[: m_leja + 1]

    spectral = FocalIntervalEstimator(min_floor=1e-12)
    dd = DividedDifferencesLog(taylor_degree=250)
    leja = LejaLogAction(divided_diff=dd)
    hpp = HutchPP(rng=42)

    estimator = LogDetEstimatorHutchPPLeja(spectral=spectral, leja_action=leja, hutchpp=hpp)

    t0 = time.perf_counter()
    est, info = estimator.estimate(Q, leja_points=Leja_X, m_leja=m_leja, m_hutchpp=m_hutchpp, tol=tol)
    t1 = time.perf_counter()

    print("trace(log(Q)) ≈", est)
    print("info:", info)
    print(f"Elapsed: {t1 - t0:.3f} s")
    print("Max adaptive Leja degree used:", info["leja_m_max"])